### 3 Kolla nya ifkdb 
* https://ifkdb.se/listor/allaspelare
* [#32](https://github.com/salgo60/ifkdb/issues/32)
* denna notebook [32_ifkdb_all.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/32_ifkdb_all.ipynb)
* lista
   * [Notebook/ifkdb_mixnmatch.csv](https://github.com/salgo60/ifkdb/blob/main/Notebook/ifkdb_mixnmatch.csv)
   * [csv](https://salgo60.github.io/ifkdb/Notebook/ifkdb_mixnmatch.csv)

In [1]:
from datetime import datetime
import platform
import sys
import os

# Start timer
start_time = datetime.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")


Start time: 2026-03-03 16:52:03


In [2]:
import requests
import urllib.parse
import pandas as pd
from bs4 import BeautifulSoup
from SPARQLWrapper import SPARQLWrapper, JSON

# --------------------------------------------------
# 1. HÄMTA ALLA SPELARE FRÅN IFKDB
# --------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "IFKDB-Match/1.0 (salgo60@msn.com)"
})

url = "https://ifkdb.se/listor/allaspelare"
r = session.get(url)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

players = {}

for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/spelare/" in href:
        full_id = href.split("/spelare/")[1]
        full_id = urllib.parse.unquote(full_id)
        players[full_id] = a.text.strip()

print("Antal unika spelare:", len(players))


# --------------------------------------------------
# 2. SKAPA dfPlayers + NORMALISERA NAMN
# --------------------------------------------------

def normalize_name(name):
    parts = name.strip().split(" ", 1)
    if len(parts) != 2:
        return name.title()
    surname, firstname = parts
    return f"{firstname.title()} {surname.title()}"

dfPlayers = pd.DataFrame([
    {
        "full_id": pid,
        "numeric_id": pid.split("_")[-1],
        "name": normalize_name(name)
    }
    for pid, name in players.items()
])

print("Verifierade spelare:", len(dfPlayers))


# --------------------------------------------------
# 3. HÄMTA ALLA P11905 FRÅN WIKIDATA
# --------------------------------------------------

endpoint = "https://query.wikidata.org/sparql"

query = """
SELECT ?item ?ifkdb WHERE {
  ?item wdt:P11905 ?ifkdb .
}
"""

sparql = SPARQLWrapper(endpoint)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent", "IFKDB-Match/1.0 (your_email@example.com)")

results = sparql.query().convert()

rows = []
for result in results["results"]["bindings"]:
    item = result["item"]["value"].split("/")[-1]
    ifkdb = result["ifkdb"]["value"]
    numeric_id = ifkdb.split("_")[-1]
    rows.append({
        "item": item,
        "numeric_id": numeric_id
    })

wd = pd.DataFrame(rows)

print("Wikidata entries:", len(wd))


# --------------------------------------------------
# 4. MATCHA PÅ NUMERISK ID (DETERMINISTISKT)
# --------------------------------------------------

matched = dfPlayers.merge(
    wd,
    on="numeric_id",
    how="left"
)

print("Matchade:", matched["item"].notna().sum())
print("Saknas:", matched["item"].isna().sum())


# --------------------------------------------------
# 5. SKAPA MIX'N'MATCH CSV
# --------------------------------------------------

matched["desc"] = "IFK Göteborg A-lagsspelare enligt ifkdb.se"
matched["type"] = "Q5"
matched["q"] = matched["item"]  # Manual match om finns

dfMix = matched[["full_id", "name", "desc", "type", "q"]]
dfMix.columns = ["id", "name", "desc", "type", "q"]

dfMix.to_csv("ifkdb_mixnmatch.csv", index=False, encoding="utf-8")

print("CSV skapad: ifkdb_mixnmatch.csv")

Antal unika spelare: 1013
Verifierade spelare: 1013
Wikidata entries: 582
Matchade: 579
Saknas: 434
CSV skapad: ifkdb_mixnmatch.csv


In [5]:
# --------------------------------------------------
# 6. AVANCERAD HTML-RAPPORT
# --------------------------------------------------

import urllib.parse

total = len(matched)
matched_count = matched["item"].notna().sum()
unmatched = matched[matched["item"].isna()].copy()
unmatched_count = len(unmatched)

match_percent = round((matched_count / total) * 100, 2)

print("Totalt:", total)
print("Matchade:", matched_count)
print("Saknade:", unmatched_count)


# --------------------------------------------------
# DUBBELTTEST (samma namn flera gånger i IFKDB)
# --------------------------------------------------

name_counts = matched["name"].value_counts()
duplicate_names = set(name_counts[name_counts > 1].index)

unmatched["duplicate"] = unmatched["name"].isin(duplicate_names)

# --------------------------------------------------
# SKAPA LÄNKER
# --------------------------------------------------

unmatched["ifkdb_url"] = (
    "https://ifkdb.se/spelare/" + unmatched["full_id"]
)

unmatched["wd_search"] = unmatched["name"].apply(
    lambda n: "https://www.wikidata.org/w/index.php?search=" +
              urllib.parse.quote(n)
)

# --------------------------------------------------
# GENERERA HTML
# --------------------------------------------------

html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>IFKDB – Wikidata matchrapport</title>

<style>
body {{ font-family: Arial, sans-serif; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px; }}
th {{ background-color: #f2f2f2; cursor: pointer; }}
tr:nth-child(even) {{ background-color: #f9f9f9; }}
.duplicate {{ background-color: #ffe0e0; }}
.summary {{ margin-bottom: 20px; }}
a {{ text-decoration: none; color: #0645AD; }}
</style>

<script>
// Enkel tabellsortering
function sortTable(n) {{
  var table = document.getElementById("reportTable");
  var rows = Array.from(table.rows).slice(1);
  var asc = table.getAttribute("data-sort") !== "asc";
  rows.sort(function(a, b) {{
    var x = a.cells[n].innerText.toLowerCase();
    var y = b.cells[n].innerText.toLowerCase();
    return asc ? x.localeCompare(y) : y.localeCompare(x);
  }});
  table.setAttribute("data-sort", asc ? "asc" : "desc");
  rows.forEach(row => table.appendChild(row));
}}
</script>

</head>
<body>

<h2>IFKDB → Wikidata matchrapport</h2>

<div class="summary">
<p><strong>Totalt:</strong> {total}</p>
<p><strong>Matchade:</strong> {matched_count}</p>
<p><strong>Saknade:</strong> {unmatched_count}</p>
<p><strong>Matchningsgrad:</strong> {match_percent}%</p>
<p><strong>GitHub Issue:</strong>
<a href="https://github.com/salgo60/ifkdb/issues/32" target="_blank">
Issue #32
</a></p>
</div>

<table id="reportTable" data-sort="asc">
<tr>
<th onclick="sortTable(0)">ID</th>
<th onclick="sortTable(1)">Namn</th>
<th>IFKDB</th>
<th>Wikidata-sök</th>
</tr>
"""

for _, row in unmatched.iterrows():

    duplicate_class = "duplicate" if row["duplicate"] else ""

    html += f"""
<tr class="{duplicate_class}">
<td>{row['full_id']}</td>
<td>{row['name']}</td>
<td><a href="{row['ifkdb_url']}" target="_blank">Öppna</a></td>
<td><a href="{row['wd_search']}" target="_blank">Sök</a></td>
</tr>
"""

html += """
</table>

<p style="margin-top:20px;">
Röd markering = potentiell namndubblett i IFKDB.
</p>

</body>
</html>
"""

with open("ifkdb_wikidata_report.html", "w", encoding="utf-8") as f:
    f.write(html)

print("HTML-rapport skapad: ifkdb_wikidata_report.html")

Totalt: 1013
Matchade: 579
Saknade: 434
HTML-rapport skapad: ifkdb_wikidata_report.html


In [3]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")


--- Execution Summary ---
End time:      2026-03-03 16:52:05
Duration:      0:00:02.272284
Total seconds: 2.27
Python ver:    3.12.2
Platform:      Darwin 25.3.0
Process ID:    57438
